In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from datetime import datetime

processed_dir = "data/processed"
for f in os.listdir(processed_dir):
    if f.endswith(".index"):
        path = os.path.join(processed_dir, f)
        mtime = os.path.getmtime(path)
        print(f"{f}: Last Modified -> {datetime.fromtimestamp(mtime)}")

faiss_ft_dup_only.index: Last Modified -> 2026-06-28 18:17:48.188067
baseline.index: Last Modified -> 2026-05-31 14:28:00.542630
faiss_ft_hard_neg.index: Last Modified -> 2026-06-28 18:18:35.082730
faiss_title_body.index: Last Modified -> 2026-06-28 18:16:18.826334
faiss_ft_all_pairs.index: Last Modified -> 2026-06-28 18:17:03.258776


In [3]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('models/ft-all-pairs')
print(m.get_sentence_embedding_dimension())

/Users/apple/ml-env/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


768


/var/folders/rt/sfphsczs46qc5ttq57cjgv200000gn/T/ipykernel_42885/3820028144.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(m.get_sentence_embedding_dimension())


In [4]:
import faiss, numpy as np

combined_embeddings = np.load('data/processed/combined_embeddings.npy').astype('float32')
faiss.normalize_L2(combined_embeddings)
index = faiss.IndexFlatIP(combined_embeddings.shape[1])
index.add(combined_embeddings)
faiss.write_index(index, 'data/processed/faiss_title_body.index')

In [5]:
import sys
sys.path.insert(0, '..')   # so we can import from src/

import numpy as np
import pandas as pd
import faiss
import json
from pathlib import Path

from src.evaluate import (
    make_eval_splits,
    evaluate_retrieval,
    evaluate_by_sport,
    save_metrics,
)
from src.retrieve import SemanticRetriever
from src.train import mine_hard_negatives, combine_title_body

# --- Load data (same split used in Weeks 4 & 5) ---
questions   = pd.read_csv('data/processed/questions.csv')
pairs       = pd.read_csv('data/processed/pairs.csv')
train_pairs = pd.read_csv('data/processed/train_pairs.csv')
eval_pairs  = pd.read_csv('data/processed/eval_pairs.csv')

print(f'Questions  : {len(questions)}')
print(f'Train pairs: {len(train_pairs)}')
print(f'Eval pairs : {len(eval_pairs)}')

Questions  : 6107
Train pairs: 620
Eval pairs : 156


In [6]:
# Spot-check: how many duplicate vs linked pairs in the training split?
vc = train_pairs['LinkTypeId'].value_counts()
print('Train pairs by type:')
print(f'  LinkTypeId 3 (duplicate): {vc.get(3, 0)}')
print(f'  LinkTypeId 1 (linked)   : {vc.get(1, 0)}')
print()
print('Duplicate pairs are higher-quality training signal.')
print('The duplicates-only ablation tests whether quality beats quantity.')

Train pairs by type:
  LinkTypeId 3 (duplicate): 24
  LinkTypeId 1 (linked)   : 596

Duplicate pairs are higher-quality training signal.
The duplicates-only ablation tests whether quality beats quantity.


In [7]:
hard_neg_path = Path('data/processed/hard_negatives.csv')

if not hard_neg_path.exists():
    hard_negatives = mine_hard_negatives(
        questions=questions,
        train_pairs=train_pairs,
        model_name='all-MiniLM-L6-v2',
        top_n=20,
        output_path=str(hard_neg_path),
    )
else:
    hard_negatives = pd.read_csv(hard_neg_path)
    print(f'Loaded {len(hard_negatives)} hard negatives from disk.')

hard_negatives.head()

Loaded 24 hard negatives from disk.


,anchor_id,positive_id,negative_id
0,29426,2682,17192
1,14570,255,1813
2,29016,16853,17998
3,25389,3,24554
4,12010,1335,3901


In [8]:
# Spot-check a few mined triplets — read them, do they make sense?
id_map = {int(qid): i for i, qid in enumerate(questions['Id'])}

def show_triplet(row):
    def title(qid):
        idx = id_map.get(int(qid))
        return questions.iloc[idx]['Title'] if idx is not None else '??'
    print(f"  Anchor  : {title(row['anchor_id'])}")
    print(f"  Positive: {title(row['positive_id'])}")
    print(f"  Hard neg: {title(row['negative_id'])}")
    print()

print('=== 5 mined triplets ===')
for _, row in hard_negatives.head(5).iterrows():
    show_triplet(row)

=== 5 mined triplets ===
  Anchor  : Caught by wicket keaper
  Positive: What is the rule when the ball touches both the bat and the batsman's body?
  Hard neg: Running between the wickets rule

  Anchor  : What are the criteria for deciding which sports to include in the Olympic curriculum?
  Positive: How does a sport become an Olympic Sport?
  Hard neg: Team sports in olympics

  Anchor  : Qualification of tournament
  Positive: How is the "best loser" determined in football?
  Hard neg: Tiebreakers for best loser in a single elimination playoff

  Anchor  : Arsenal - Rapid Wein. Why wasn't this offside?
  Positive: How is offside determined in football?
  Hard neg: Salah's goal assisted by Alisson Becker was not offside why?

  Anchor  : Why do people grunt while playing tennis?
  Positive: If grunting is not related to performance, why do tennis players do it excessively?
  Hard neg: Why do these players rub their hands on the table near the net?



In [9]:
from sentence_transformers import SentenceTransformer

def build_and_save_index(model_path: str, questions: pd.DataFrame, index_save_path: str):
    """
    Encode corpus with a (possibly fine-tuned) model using combine_title_body(),
    build a FAISS index, and save it.
    """
    print(f'Building index for: {model_path}')
    model = SentenceTransformer(model_path)
    texts = [combine_title_body(row) for _, row in questions.iterrows()]
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, index_save_path)
    print(f'  → {index_save_path}  ({index.ntotal} vectors)\n')


VARIANTS = [
    ('ft-all-pairs',       'models/ft-all-pairs',       'data/processed/faiss_ft_all_pairs.index'),
    ('ft-duplicates-only', 'models/ft-duplicates-only', 'data/processed/faiss_ft_dup_only.index'),
    ('ft-hard-negatives',  'models/ft-hard-negatives',  'data/processed/faiss_ft_hard_neg.index'),
]

for name, model_path, index_path in VARIANTS:
    if Path(model_path).exists():
        build_and_save_index(model_path, questions, index_path)
    else:
        print(f'[skip] {name} — model folder not found at {model_path}')

Building index for: models/ft-all-pairs


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

  → data/processed/faiss_ft_all_pairs.index  (6107 vectors)

Building index for: models/ft-duplicates-only


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

  → data/processed/faiss_ft_dup_only.index  (6107 vectors)

Building index for: models/ft-hard-negatives


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

  → data/processed/faiss_ft_hard_neg.index  (6107 vectors)



In [10]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

def make_retriever(model_path: str, index_path: str) -> SemanticRetriever:
    index = faiss.read_index(index_path)
    # If the model_path is a local folder, it loads fine-tuned weights. 
    # If it's 'all-MiniLM-L6-v2', it downloads the base model from Hugging Face.
    model = SentenceTransformer(model_path)
    
    # Now this matches our newly updated class!
    return SemanticRetriever(index=index, questions_df=questions, model=model)


# --- Collect results ---
all_results = {}

# Removed the ../ to fix the pathing issue
baseline_index_path = 'data/processed/faiss_title_body.index'

if Path(baseline_index_path).exists():
    baseline_retriever = make_retriever('all-mpnet-base-v2', baseline_index_path)
    all_results['baseline (title+body)'] = evaluate_retrieval(
        baseline_retriever, eval_pairs.iloc[:, :3].values, questions
    )
else:
    print('[skip] Baseline FAISS index not found — run build_index.py first')
    # Insert known Week 5 numbers as a reference
    all_results['baseline (title+body)'] = {
        'recall@1': 0.2792, 'recall@5': 0.4740, 'recall@10': 0.5649, 'mrr': 0.3633
    }

# Fine-tuned variants
for name, model_path, index_path in VARIANTS:
    if Path(model_path).exists() and Path(index_path).exists():
        retriever = make_retriever(model_path, index_path)
        all_results[name] = evaluate_retrieval(
            retriever, eval_pairs.iloc[:, :3].values, questions
        )
        print(f'{name}: {all_results[name]}')
    else:
        print(f'[skip] {name} — model or index not found')

print('\nDone.')

ft-all-pairs: {'recall@1': 0.2922, 'recall@5': 0.5649, 'recall@10': 0.6883, 'mrr': 0.4144, 'total': 154}
ft-duplicates-only: {'recall@1': 0.2792, 'recall@5': 0.5195, 'recall@10': 0.6104, 'mrr': 0.3856, 'total': 154}
ft-hard-negatives: {'recall@1': 0.2792, 'recall@5': 0.5195, 'recall@10': 0.6104, 'mrr': 0.3854, 'total': 154}

Done.


In [11]:
# --- Pretty comparison table ---
metrics_order = ['recall@1', 'recall@5', 'recall@10', 'mrr']

rows = []
for model_name, metrics in all_results.items():
    row = {'model': model_name}
    row.update({k: metrics.get(k, '—') for k in metrics_order})
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('model')
print('\n=== Results vs Baseline ===')
print(results_df.to_string())

# Save
save_metrics(all_results, 'results/week6_metrics.json')


=== Results vs Baseline ===
                       recall@1  recall@5  recall@10     mrr
model                                                       
baseline (title+body)    0.2792    0.5195     0.6104  0.3857
ft-all-pairs             0.2922    0.5649     0.6883  0.4144
ft-duplicates-only       0.2792    0.5195     0.6104  0.3856
ft-hard-negatives        0.2792    0.5195     0.6104  0.3854
Saved metrics to results/week6_metrics.json


In [12]:
# Extract just the two ablation models for a focused comparison
ablation_keys = ['baseline (title+body)', 'ft-all-pairs', 'ft-duplicates-only']
ablation_rows = []

for key in ablation_keys:
    if key in all_results:
        m = all_results[key]
        ablation_rows.append({
            'model': key,
            'n_train_pairs': '—' if 'baseline' in key else (
                len(train_pairs[train_pairs['LinkTypeId'] == 3]) if 'dup' in key
                else len(train_pairs)
            ),
            'recall@5': m.get('recall@5', '—'),
            'recall@10': m.get('recall@10', '—'),
            'mrr': m.get('mrr', '—'),
        })

ablation_df = pd.DataFrame(ablation_rows).set_index('model')
print('=== Ablation: pair quality vs quantity ===')
print(ablation_df.to_string())

print()
print('Interpretation:')
print('  If ft-all-pairs > ft-duplicates-only: quantity wins despite noise.')
print('  If ft-duplicates-only > ft-all-pairs: quality wins despite fewer pairs.')
print('  Either finding is worth writing up — neither is a foregone conclusion.')

=== Ablation: pair quality vs quantity ===
                      n_train_pairs  recall@5  recall@10     mrr
model                                                           
baseline (title+body)             —    0.5195     0.6104  0.3857
ft-all-pairs                    620    0.5649     0.6883  0.4144
ft-duplicates-only               24    0.5195     0.6104  0.3856

Interpretation:
  If ft-all-pairs > ft-duplicates-only: quantity wins despite noise.
  If ft-duplicates-only > ft-all-pairs: quality wins despite fewer pairs.
  Either finding is worth writing up — neither is a foregone conclusion.


In [17]:
# Run sport breakdown for baseline and best fine-tuned model
sport_results = {}

if 'baseline (title+body)' in all_results:
    
    sport_results['baseline'] = evaluate_by_sport(baseline_retriever, eval_pairs.iloc[:, :3].values, questions)
    
# Use ft-all-pairs as the primary fine-tuned model for sport breakdown
ft_model_path = 'models/ft-all-pairs'
ft_index_path = 'data/processed/faiss_ft_all_pairs.index'
if Path(ft_model_path).exists():
    ft_retriever = make_retriever(ft_model_path, ft_index_path)
    sport_results['ft-all-pairs'] = evaluate_by_sport(ft_retriever, eval_pairs.iloc[:, :3].values, questions)

# Print side-by-side
if sport_results:
    sports = sorted(
        set(s for r in sport_results.values() for s in r.keys()),
        key=lambda s: sport_results.get('baseline', {}).get(s, {}).get('n_pairs', 0),
        reverse=True,
    )

    print(f'{"Sport":<22} {"n_pairs":>8}', end='')
    for model_name in sport_results:
        print(f'  {model_name:>18} recall@5', end='')
    print()
    print('-' * 80)

    for sport in sports:
        n = sport_results.get('baseline', {}).get(sport, {}).get('n_pairs', '—')
        print(f'{sport:<22} {str(n):>8}', end='')
        for model_name in sport_results:
            r5 = sport_results[model_name].get(sport, {}).get('recall@5', '—')
            print(f'  {"{:.4f}".format(r5) if isinstance(r5, float) else r5:>27}', end='')
        print()

    save_metrics(sport_results, 'results/week6_sport_breakdown.json')

Sport                   n_pairs            baseline recall@5        ft-all-pairs recall@5
--------------------------------------------------------------------------------
rules                        62                       0.5484                       0.5806
football                     28                       0.3929                       0.4286
cricket                      12                       0.3333                       0.5000
american-football             9                       0.6667                       0.7778
tennis                        8                       0.7500                       0.8750
baseball                      7                       0.5714                       0.5714
equipment                     6                       0.1667                       0.1667
trivia                        5                       0.8000                       0.8000
training                      4                       1.0000                       1.0000
technique          

In [18]:
question_id_to_idx = {int(qid): i for i, qid in enumerate(questions['Id'])}
wins = []

for post_id, related_post_id, _ in eval_pairs.iloc[:, :3].values:
    post_id, related_post_id = int(post_id), int(related_post_id)
    if post_id not in question_id_to_idx or related_post_id not in question_id_to_idx:
        continue
    query_text = combine_title_body(questions.iloc[question_id_to_idx[post_id]])

    base_ids = [r['question_id'] for r in baseline_retriever.search(query_text, top_k=6) if r['question_id'] != post_id][:5]
    ft_ids   = [r['question_id'] for r in ft_retriever.search(query_text, top_k=6) if r['question_id'] != post_id][:5]

    if related_post_id in ft_ids and related_post_id not in base_ids:
        wins.append({
            'query': questions.iloc[question_id_to_idx[post_id]]['Title'],
            'target': questions.iloc[question_id_to_idx[related_post_id]]['Title'],
        })

print(f"{len(wins)} cases where fine-tuning fixed a baseline miss")
for w in wins[:8]:
    print(f"  Q: {w['query']}  ->  {w['target']}")

10 cases where fine-tuning fixed a baseline miss
  Q: How is the WASP calculated in cricket?  ->  How is the Duckworth-Lewis Method applied?
  Q: Position of the Defense in Soccer Offside  ->  How is offside determined in football?
  Q: Why players showing a T-signal for umpire decision review system?  ->  Umpire Decision Review System in cricket
  Q: Can a player hold the ball under their body or legs in football?  ->  Squeezing the ball between two feet and hopping across the field
  Q: Difference between three technique and five technique defensive linemen  ->  What are the disadvantages of the wide nine defensive scheme?
  Q: Red card during substitution in Football  ->  Can a player get his second yellow card while being replaced? What happens then?
  Q: Can you block and hit the ball over the net in volleyball?  ->  What constitutes a legal block in volleyball in regard to it not counting as the first hit?
  Q: Which female football players have highest goal per game ratio for na